# Kaggriculture submission v3: cash-safe staple expansion

V3 preserves the valid `main.py` / `agent(obs)` contract and addresses the main
economic weaknesses observed in v2:

- buys only the NE quadrant, using an exact cash-safe opening;
- expands from 25 to 50 managed plots without adding another premium field;
- uses NE mainly for wheat and carrots, whose glut curves are safer;
- makes melons an initial-wave-only crop, then rotates those plots evenly into
  wheat and carrots;
- sells the first melon cohort immediately instead of planting a second wave
  before the market reflects the first harvest;
- includes carried worker inventory in shed-pressure decisions;
- prioritizes critically unwatered crops and tasks that remain reachable before
  the end of the day.

The submitted agent is deterministic, self-contained, and performs no network or
filesystem access during an episode.

Official references: [competition overview](https://www.kaggle.com/competitions/kaggriculture/overview), [rules](https://www.kaggle.com/competitions/kaggriculture/rules).

## 1. Install the same environment version used for local validation

This installation is only for building/testing the notebook. The submitted `main.py` itself has no external dependencies.

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments==1.32.7"

## 2. Write the required v3 entrypoint

The strategy performs a staged 47-plot bootstrap before transitioning to a
50-plot steady plan. Seven hands are used on setup day, six while cash is tight,
and seven after the first staple harvest funds operations. Premium crops are
price- and town-gated; the expanded field remains staple-heavy.

In [ ]:
%%writefile main.py
"""Kaggriculture v3: cash-safe NE staple expansion."""


PASS = ["PASS"]
MAX_MARKET_ORDERS = 10
PRESSURE_STOCK = 70
EMERGENCY_STOCK = 90
LIQUIDATION_DAY = 25
FINAL_FARM_DAY = 28


CROPS = {
    "WHEAT": {
        "seed_cost": 10,
        "base_price": 25,
        "first_yield_day": 2,
        "harvest_day": 4,
        "last_plant_day": 24,
        "ongoing": False,
    },
    "CARROT": {
        "seed_cost": 20,
        "base_price": 35,
        "first_yield_day": 2,
        "harvest_day": 3,
        "last_plant_day": 25,
        "ongoing": False,
    },
    "TOMATO": {
        "seed_cost": 50,
        "base_price": 60,
        "first_yield_day": 8,
        "harvest_day": 8,
        "last_plant_day": 18,
        "ongoing": True,
    },
    "STRAWBERRY": {
        "seed_cost": 100,
        "base_price": 120,
        "first_yield_day": 10,
        "harvest_day": 10,
        "last_plant_day": 16,
        "ongoing": True,
    },
    "MELON": {
        "seed_cost": 80,
        "base_price": 250,
        "first_yield_day": 10,
        "harvest_day": 10,
        "last_plant_day": 1,
        "ongoing": False,
    },
}


# x, y, preferred, fallback, initial bootstrap crop.  NW preserves v2's first
# premium wave, but its twelve melon slots split into six wheat and six carrot
# fallbacks after that wave.  NE starts as 20 wheat + 2 carrots and transitions
# to a safer steady mix of 15 W / 6 C / 2 T / 2 S.
PLOT_SLOTS = (
    # NW
    (0, 0, "MELON", "CARROT", "MELON"),
    (1, 0, "STRAWBERRY", "WHEAT", "STRAWBERRY"),
    (2, 0, "MELON", "WHEAT", "MELON"),
    (3, 0, "CARROT", "WHEAT", "CARROT"),
    (4, 0, "MELON", "CARROT", "MELON"),
    (0, 1, "WHEAT", "CARROT", "WHEAT"),
    (1, 1, "MELON", "WHEAT", "MELON"),
    (2, 1, "STRAWBERRY", "WHEAT", "STRAWBERRY"),
    (3, 1, "MELON", "CARROT", "MELON"),
    (4, 1, "TOMATO", "CARROT", "TOMATO"),
    (0, 2, "MELON", "WHEAT", "MELON"),
    (1, 2, "CARROT", "WHEAT", "CARROT"),
    (2, 2, "MELON", "CARROT", "MELON"),
    (3, 2, "STRAWBERRY", "WHEAT", "STRAWBERRY"),
    (4, 2, "MELON", "WHEAT", "MELON"),
    (0, 3, "STRAWBERRY", "WHEAT", "STRAWBERRY"),
    (1, 3, "MELON", "CARROT", "MELON"),
    (2, 3, "WHEAT", "CARROT", "WHEAT"),
    (3, 3, "MELON", "WHEAT", "MELON"),
    (4, 3, "TOMATO", "CARROT", "TOMATO"),
    (0, 4, "MELON", "CARROT", "MELON"),
    (1, 4, "CARROT", "WHEAT", "CARROT"),
    (2, 4, "STRAWBERRY", "WHEAT", "STRAWBERRY"),
    (3, 4, "WHEAT", "CARROT", "WHEAT"),
    (4, 4, "MELON", "WHEAT", "MELON"),
    # NE steady rows; bootstrap is 20 W, 2 C, then 3 idle slots.
    (5, 0, "WHEAT", "CARROT", "WHEAT"),
    (6, 0, "CARROT", "WHEAT", "CARROT"),
    (7, 0, "WHEAT", "CARROT", "WHEAT"),
    (8, 0, "WHEAT", "CARROT", "WHEAT"),
    (9, 0, "WHEAT", "CARROT", "WHEAT"),
    (5, 1, "CARROT", "WHEAT", "CARROT"),
    (6, 1, "WHEAT", "CARROT", "WHEAT"),
    (7, 1, "WHEAT", "CARROT", "WHEAT"),
    (8, 1, "CARROT", "WHEAT", "WHEAT"),
    (9, 1, "WHEAT", "CARROT", "WHEAT"),
    (5, 2, "WHEAT", "CARROT", "WHEAT"),
    (6, 2, "CARROT", "WHEAT", "WHEAT"),
    (7, 2, "WHEAT", "CARROT", "WHEAT"),
    (8, 2, "WHEAT", "CARROT", "WHEAT"),
    (9, 2, "WHEAT", "CARROT", "WHEAT"),
    (5, 3, "CARROT", "WHEAT", "WHEAT"),
    (6, 3, "WHEAT", "CARROT", "WHEAT"),
    (7, 3, "TOMATO", "WHEAT", "WHEAT"),
    (8, 3, "STRAWBERRY", "WHEAT", "WHEAT"),
    (9, 3, "WHEAT", "CARROT", "WHEAT"),
    (5, 4, "CARROT", "WHEAT", "WHEAT"),
    (6, 4, "WHEAT", "CARROT", "WHEAT"),
    (7, 4, "TOMATO", "WHEAT", None),
    (8, 4, "STRAWBERRY", "WHEAT", None),
    (9, 4, "WHEAT", "CARROT", None),
)


SELL_RULES = {
    "WHEAT": (24, 18),
    "CARROT": (18, 20),
    "TOMATO": (8, 40),
    "STRAWBERRY": (4, 90),
    "MELON": (4, 185),
}


STRAWBERRY_SHOPS = {
    "BRUNCH_SPOT",
    "ICE_CREAM_SHOP",
    "SMOOTHIE_SHOP",
    "FARMERS_MARKET",
}


def _safe_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _step_toward(position, target):
    x, y = position
    tx, ty = target
    dx = tx - x
    dy = ty - y
    if abs(dx) >= abs(dy) and dx:
        return ["EAST" if dx > 0 else "WEST"]
    if dy:
        return ["SOUTH" if dy > 0 else "NORTH"]
    return PASS


def _desired_hands(day):
    if day == 0:
        return 7
    if day <= 3:
        return 6
    if day <= FINAL_FARM_DAY:
        return 7
    return 0


def _fib(index):
    a, b = 1, 1
    for _ in range(max(0, index)):
        a, b = b, a + b
    return a


def _crop_counts(farm):
    counts = {crop: 0 for crop in CROPS}
    for row in (farm.get("tiles", []) or []):
        for tile in row:
            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                crop = tile.get("crop")
                if crop in counts:
                    counts[crop] += 1
    return counts


def _slot_crop(
    primary, fallback, bootstrap, day, market, town, opponent_counts
):
    # Bootstrap crops remain the empty-slot target through day 2.  By the time
    # initial carrots/wheat are harvested, the slot transitions to steady state.
    if day <= 2:
        return bootstrap
    if primary == "MELON":
        return fallback

    prices = (market or {}).get("prices", {}) or {}
    shops = (town or {}).get("unlocked_shops", []) or []
    candidate = primary
    price = _safe_int(prices.get(primary), CROPS[primary]["base_price"])
    opponent_count = opponent_counts.get(primary, 0)

    if primary == "STRAWBERRY":
        support = sum(shop in STRAWBERRY_SHOPS for shop in shops)
        if price < 105 and not (price >= 90 and support > 0):
            candidate = fallback
        if opponent_count >= 6:
            candidate = fallback
    elif primary == "TOMATO":
        support = sum(shop in {"PIZZA_SHOP", "FARMERS_MARKET"} for shop in shops)
        if price < 45 and not (price >= 36 and support > 0):
            candidate = fallback
    elif primary in ("WHEAT", "CARROT") and fallback in ("WHEAT", "CARROT"):
        wheat_price = _safe_int(prices.get("WHEAT"), 25)
        carrot_price = _safe_int(prices.get("CARROT"), 35)
        wheat_score = 0.80 * wheat_price - 2.5
        carrot_score = 0.75 * carrot_price - 6.7
        if primary == "WHEAT" and carrot_score > wheat_score * 1.25:
            candidate = "CARROT"
        elif primary == "CARROT" and wheat_score > carrot_score * 1.15:
            candidate = "WHEAT"

    if day <= CROPS[candidate]["last_plant_day"]:
        return candidate
    if day <= CROPS[fallback]["last_plant_day"]:
        return fallback
    if day <= CROPS["WHEAT"]["last_plant_day"]:
        return "WHEAT"
    if day <= CROPS["CARROT"]["last_plant_day"]:
        return "CARROT"
    return None


def _tile_task(tile, crop_to_plant, day, hour, step):
    if tile is None:
        if crop_to_plant is not None and hour <= 18:
            return 5, ["PLANT", crop_to_plant]
        return None
    if tile == "LOCKED" or not isinstance(tile, dict):
        return None
    if tile.get("kind") == "WEED":
        return 4, ["DIG"]
    if tile.get("kind") != "PLANT":
        return 4, ["DIG"]

    crop = tile.get("crop")
    data = CROPS.get(crop)
    if data is None:
        return 4, ["DIG"]
    age = day - _safe_int(tile.get("planted_day"), day)
    yield_units = _safe_int(tile.get("yield_units"), 0)
    watered = bool(tile.get("watered_today", False))
    missed = _safe_int(tile.get("consecutive_unwatered"), 0)
    max_step = _safe_int(tile.get("max_lifespan_step"), -1)

    if day >= FINAL_FARM_DAY and yield_units > 0 and age >= data["first_yield_day"]:
        return 0, ["HARVEST"]
    if max_step >= 0 and step >= max_step and yield_units <= 0:
        return 1, ["DIG"]
    if data["ongoing"]:
        if yield_units > 0 and age >= data["first_yield_day"]:
            return 1, ["HARVEST"]
        if not watered:
            return (0 if missed >= 1 else 3), ["WATER"]
        return None
    if yield_units > 0 and age >= data["harvest_day"]:
        if age == data["harvest_day"] and not watered:
            return 0, ["WATER"]
        return 1, ["HARVEST"]
    if not watered:
        return (0 if missed >= 1 else 3), ["WATER"]
    return None


def _build_tasks(
    farm, private, day, hour, step, market, town, opponent_counts
):
    tiles = farm.get("tiles", []) or []
    seeds_available = {
        crop: _safe_int((private.get("seeds", {}) or {}).get(crop), 0)
        for crop in CROPS
    }
    tasks = []
    for x, y, primary, fallback, bootstrap in PLOT_SLOTS:
        try:
            tile = tiles[y][x]
        except (IndexError, TypeError):
            continue
        crop = _slot_crop(
            primary, fallback, bootstrap, day, market, town, opponent_counts
        )
        task = _tile_task(tile, crop, day, hour, step)
        if task is None:
            continue
        priority, action = task
        if action[0] == "PLANT":
            crop = action[1]
            if seeds_available[crop] <= 0:
                continue
            seeds_available[crop] -= 1
        tasks.append(
            {"priority": priority, "target": (x, y), "action": action}
        )
    return tasks


def _unit_actions(
    farm, private, day, hour, step, market, town, opponent_counts
):
    positions = [tuple(farm.get("farmer", (4, 4)))]
    positions.extend(tuple(pos) for pos in (farm.get("hands", []) or []))
    actions = [PASS for _ in positions]
    tasks = _build_tasks(
        farm, private, day, hour, step, market, town, opponent_counts
    )
    available = set(range(len(positions)))
    turns_left = 24 - hour

    while tasks and available:
        choices = []
        for unit_index in available:
            ux, uy = positions[unit_index]
            for task_index, task in enumerate(tasks):
                tx, ty = task["target"]
                distance = abs(tx - ux) + abs(ty - uy)
                if distance + 1 > turns_left:
                    continue
                choices.append(
                    (
                        task["priority"],
                        distance,
                        ty,
                        tx,
                        unit_index,
                        task_index,
                    )
                )
        if not choices:
            break
        _, _, _, _, unit_index, task_index = min(choices)
        task = tasks.pop(task_index)
        available.remove(unit_index)
        if positions[unit_index] == task["target"]:
            actions[unit_index] = task["action"]
        else:
            actions[unit_index] = _step_toward(
                positions[unit_index], task["target"]
            )
    return actions


def _exposure(private):
    shed = private.get("shed", {}) or {}
    inventories = private.get("inventories", []) or []
    return sum(max(0, _safe_int(value)) for value in shed.values()) + sum(
        max(0, _safe_int(value)) for inv in inventories for value in inv.values()
    )


def _sell_orders(private, market, day):
    shed = private.get("shed", {}) or {}
    prices = (market or {}).get("prices", {}) or {}
    exposure = _exposure(private)
    pressure = exposure >= PRESSURE_STOCK
    emergency = exposure >= EMERGENCY_STOCK
    terminal = day >= LIQUIDATION_DAY
    orders = []

    # Melon has no shop demand and recovers very slowly.  Capture the first
    # high-price curve immediately instead of holding 72 units in a 100-cap shed.
    melons = _safe_int(shed.get("MELON"), 0)
    if melons >= 24 and 9 <= day <= 13:
        orders.append(["SELL", "MELON", melons])

    for item in ("STRAWBERRY", "TOMATO", "CARROT", "WHEAT", "MELON"):
        held = _safe_int(shed.get(item), 0)
        if held <= 0 or (item == "MELON" and melons >= 24 and 9 <= day <= 13):
            continue
        batch, threshold = SELL_RULES[item]
        price = _safe_int(prices.get(item), CROPS[item]["base_price"])
        if terminal or emergency or pressure or price >= threshold:
            quantity = held if terminal or emergency else min(held, batch)
            orders.append(["SELL", item, quantity])
    return orders


def _desired_seed_counts(farm, day, market, town, opponent_counts):
    wanted = {crop: 0 for crop in CROPS}
    tiles = farm.get("tiles", []) or []
    for x, y, primary, fallback, bootstrap in PLOT_SLOTS:
        try:
            tile = tiles[y][x]
        except (IndexError, TypeError):
            continue
        if tile is None or (isinstance(tile, dict) and tile.get("kind") == "WEED"):
            crop = _slot_crop(
                primary, fallback, bootstrap, day, market, town, opponent_counts
            )
            if crop is not None:
                wanted[crop] += 1
    return wanted


def _seed_orders(
    farm, private, day, market, town, opponent_counts, slots, cash
):
    if slots <= 0 or day >= FINAL_FARM_DAY:
        return []
    wanted = _desired_seed_counts(farm, day, market, town, opponent_counts)
    seeds = private.get("seeds", {}) or {}
    reserve = 50.0 if day <= 3 else 150.0
    orders = []
    for crop in ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"):
        if len(orders) >= slots:
            break
        missing = wanted[crop] - _safe_int(seeds.get(crop), 0)
        if missing <= 0:
            continue
        cost = CROPS[crop]["seed_cost"]
        affordable = max(0, int((cash - reserve) // cost))
        quantity = min(missing, affordable)
        if quantity <= 0:
            continue
        orders.append(["BUY_SEED", crop, quantity])
        cash -= quantity * cost
    return orders


def _market_orders(
    farm, private, market, town, day, hour, opponent_counts
):
    unlocked = farm.get("unlocked_quadrants", []) or []

    # Exact cash-safe bootstrap: $1000 land + $33 hands + $1460 M/S seeds.
    if day == 0 and hour == 0 and "NE" not in unlocked:
        return [
            ["BUY_LAND"],
            ["HIRE"],
            ["HIRE"],
            ["HIRE"],
            ["HIRE"],
            ["HIRE"],
            ["HIRE"],
            ["HIRE"],
            ["BUY_SEED", "MELON", 12],
            ["BUY_SEED", "STRAWBERRY", 5],
        ]

    # The remaining bootstrap seeds cost $430 and are bought after land/hiring.
    if day == 0 and hour == 1:
        return [
            ["BUY_SEED", "WHEAT", 23],
            ["BUY_SEED", "CARROT", 5],
            ["BUY_SEED", "TOMATO", 2],
        ]

    desired_hands = _desired_hands(day)
    current_hands = len(farm.get("hands", []) or [])
    missing_hands = max(0, desired_hands - current_hands)
    sale_orders = _sell_orders(private, market, day)
    orders = []
    cash = float(farm.get("money", 0))

    # A first-wave melon dump must stay at order index 0.  If cash is too low to
    # hire on day 4, one available sale similarly funds the following HIREs.
    lead_sale = None
    hire_cost = sum(_fib(_safe_int(farm.get("hires_today"), 0) + i) for i in range(missing_hands))
    if sale_orders and (
        sale_orders[0][1] == "MELON" or cash < hire_cost
    ):
        lead_sale = sale_orders.pop(0)
        orders.append(lead_sale)

    for _ in range(missing_hands):
        if len(orders) >= MAX_MARKET_ORDERS:
            break
        # A preceding sale is deliberately allowed to fund these orders.
        orders.append(["HIRE"])

    for order in sale_orders:
        if len(orders) >= MAX_MARKET_ORDERS:
            break
        orders.append(order)

    free_slots = MAX_MARKET_ORDERS - len(orders)
    orders.extend(
        _seed_orders(
            farm,
            private,
            day,
            market,
            town,
            opponent_counts,
            free_slots,
            cash,
        )
    )
    return orders[:MAX_MARKET_ORDERS]


def agent(obs):
    """Required Kaggle entrypoint."""
    try:
        farms = obs.get("farms", []) or []
        player = _safe_int(obs.get("player"), 0)
        private = obs.get("private", {}) or {}
        if player < 0 or player >= len(farms):
            return {"farmer": PASS, "hands": [], "market": []}
        farm = farms[player]
        opponent = farms[1 - player] if len(farms) == 2 else {}
        opponent_counts = _crop_counts(opponent)
        day = _safe_int(obs.get("day"), 0)
        hour = _safe_int(obs.get("hour"), 0)
        step = _safe_int(obs.get("step"), day * 24 + hour)
        market = obs.get("market", {}) or {}
        town = obs.get("town", {}) or {}
        market_orders = _market_orders(
            farm, private, market, town, day, hour, opponent_counts
        )

        if day > FINAL_FARM_DAY:
            farmer_action = PASS
            hand_actions = [PASS for _ in (farm.get("hands", []) or [])]
        else:
            unit_actions = _unit_actions(
                farm,
                private,
                day,
                hour,
                step,
                market,
                town,
                opponent_counts,
            )
            farmer_action = unit_actions[0] if unit_actions else PASS
            hand_actions = unit_actions[1:]

        return {
            "farmer": farmer_action,
            "hands": hand_actions,
            "market": market_orders,
        }
    except Exception:
        hands = []
        try:
            farms = obs.get("farms", []) or []
            player = _safe_int(obs.get("player"), 0)
            if 0 <= player < len(farms):
                hands = [PASS for _ in (farms[player].get("hands", []) or [])]
        except Exception:
            hands = []
        return {"farmer": PASS, "hands": hands, "market": []}

## 3. Check the entrypoint and action contract

This catches the filename/function mismatch that the tutorial's in-memory callable test misses.

In [ ]:
import ast
import importlib.util
import json
from pathlib import Path

main_path = Path("main.py")
assert main_path.is_file(), "main.py was not created"

tree = ast.parse(main_path.read_text(encoding="utf-8"), filename="main.py")
function_names = {node.name for node in tree.body if isinstance(node, ast.FunctionDef)}
assert "agent" in function_names, "main.py must expose def agent(obs)"

spec = importlib.util.spec_from_file_location("submission_agent", main_path)
submission_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_agent)
assert callable(submission_agent.agent)

tiles = [
    [None if x < 5 and y < 5 else "LOCKED" for x in range(10)]
    for y in range(10)
]
dummy_obs = {
    "player": 0,
    "day": 0,
    "hour": 0,
    "farms": [{
        "money": 3000,
        "tiles": tiles,
        "farmer": [4, 4],
        "hands": [],
        "unlocked_quadrants": ["NW"],
        "hires_today": 0,
    }],
    "private": {"shed": {}, "seeds": {}, "inventories": [{}]},
    "market": {"inventory": {}, "prices": {}},
    "town": {"unlocked_shops": []},
}

action = submission_agent.agent(dummy_obs)
assert set(action) == {"farmer", "hands", "market"}
assert isinstance(action["farmer"], list) and action["farmer"]
assert isinstance(action["hands"], list)
assert isinstance(action["market"], list) and len(action["market"]) <= 10
json.dumps(action)
print("Entrypoint and JSON action contract: OK")
print(action)

## 4. Run full file-loader validation games

These tests run the exact `main.py` filepath for all 720 turns. Self-play mirrors
Kaggle's Validation Episode; `starter` and `random` add independent smoke tests.
Rewards shown here validate execution only and are not leaderboard predictions.

In [ ]:
from kaggle_environments import make

opponents = ["main.py", "starter", "random"]
for index, opponent in enumerate(opponents):
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": 20260921 + index},
        debug=True,
    )
    env.run(["main.py", opponent])
    final = env.steps[-1]
    statuses = [state.status for state in final]
    rewards = [state.reward for state in final]
    assert statuses == ["DONE", "DONE"], (opponent, statuses)
    print(f"vs {opponent:8s}: statuses={statuses}, rewards={rewards}")

print("V3 full 720-turn file-loader validation: OK")

## 5. Build and verify the submission archive

The member name must be exactly `main.py`, with no enclosing directory.

In [ ]:
import hashlib
import tarfile
from pathlib import Path

archive_path = Path("submission.tar.gz")
with tarfile.open(archive_path, "w:gz") as archive:
    archive.add("main.py", arcname="main.py", recursive=False)

with tarfile.open(archive_path, "r:gz") as archive:
    members = archive.getnames()
    assert members == ["main.py"], members
    archived_source = archive.extractfile("main.py").read()

assert archived_source == Path("main.py").read_bytes()
size_mib = archive_path.stat().st_size / (1024 * 1024)
assert size_mib < 100, f"Archive is too large: {size_mib:.2f} MiB"
sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()

print(f"Created: {archive_path.resolve()}")
print(f"Members: {members}")
print(f"Size: {size_mib:.4f} MiB")
print(f"SHA-256: {sha256}")

## 6. Submit v3

1. Attach the **Kaggriculture** competition and accept its rules.
2. Run every cell and save a successful notebook version.
3. Confirm the full validation finishes with `DONE/DONE` and the archive cell
   reports `Members: ['main.py']`.
4. Click **Submit to competition** and select this version's
   `submission.tar.gz` output.